In [ ]:
# ==================== 配置开关 ====================
# 版本控制：当前 notebook 所在的文件夹即视为版本根目录
# 你只需要将 notebook 复制到新的版本文件夹（如 Comment_ver0.2）中即可
# 以下开关控制是否强制重新执行耗时步骤
FORCE_RECOMPUTE_FEATURES = True   # 是否强制重新生成特征（若已有特征文件，设为 False 可跳过）
FORCE_RETRAIN = True              # 是否强制重新训练模型
FORCE_RECREATE_FIGS = True        # 是否强制重新绘制图片

# ==================== 路径设置 ====================
import os
from pathlib import Path

# 获取当前 notebook 所在目录（即版本根目录）
VERSION_DIR = Path.cwd()

# 子目录路径
FIG_DIR = VERSION_DIR / "fig"
MODEL_DIR = VERSION_DIR / "model"
FEATURE_DIR = VERSION_DIR / "featured_data"

# 数据目录（假设项目根目录为当前 notebook 所在目录的上级，或根据实际情况调整）：
# 如果是colab，datadir设定为VERSION_DIR / "data"，然后新建一个文件夹上传数据；
# 如果是本地，数据要放在根文件夹里，在根文件夹创建一个data文件夹把数据放进去，notebook放在版本目录里面，避免重放很多份数据
# 本地数据组织：根文件夹-版本文件夹（version）-notebook以及其他模型内容及结果
#                     -数据文件夹（data）-三个csv
# 这里假设数据存放在 notebook 所在目录的上层 data 文件夹中
ROOT_DIR = VERSION_DIR.parent
DATA_DIR = ROOT_DIR / "data"

# 数据文件路径
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

# 创建目录（若不存在）
for dir_path in [FIG_DIR, MODEL_DIR, FEATURE_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# ==================== 打印确认信息 ====================
print(f"当前版本文件夹: {VERSION_DIR}")
print(f"图片保存目录: {FIG_DIR}")
print(f"模型保存目录: {MODEL_DIR}")
print(f"特征数据保存目录: {FEATURE_DIR}")
print(f"训练数据路径: {TRAIN_PATH}")
print(f"测试数据路径: {TEST_PATH}")
print(f"提交样例路径: {SAMPLE_SUBMISSION_PATH}")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TextStatistics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.memory", "6g") \
    .getOrCreate()

In [ ]:
# spark.stop()

In [ ]:
class ReviewPrepare:
    """
    评论数据预处理与文本特征提取（Word2Vec + TF-IDF）
    负责：加载数据、文本清洗、分词、停用词移除、Word2Vec 训练与转换、TF-IDF 训练与转换、
         特征保存、特征分析与可视化（可选）
    """

    def __init__(self, spark, train_path, test_path, feature_dir, model_dir, fig_dir,
                 force_recompute=False, force_retrain=False, force_recreate_figs=False):
        """
        初始化参数
        :param spark: SparkSession
        :param train_path: 训练集 CSV 路径
        :param test_path: 测试集 CSV 路径
        :param feature_dir: 特征保存目录（Path 对象）
        :param model_dir: 模型保存目录（Path 对象）
        :param fig_dir: 图表保存目录（Path 对象）
        :param force_recompute: 强制重新生成特征（若已存在则跳过）
        :param force_retrain: 强制重新训练 Word2Vec 和 TF-IDF 模型
        :param force_recreate_figs: 强制重新绘制图表
        """
        self.spark = spark
        self.train_path = train_path
        self.test_path = test_path
        self.feature_dir = feature_dir
        self.model_dir = model_dir
        self.fig_dir = fig_dir
        self.force_recompute = force_recompute
        self.force_retrain = force_retrain
        self.force_recreate_figs = force_recreate_figs

        # 文件路径
        self.train_feature_path = self.feature_dir / "train_features.parquet"
        self.test_feature_path = self.feature_dir / "test_features.parquet"
        self.word2vec_model_path = self.model_dir / "word2vec.model"
        self.tfidf_cv_model_path = self.model_dir / "tfidf_cv.model"
        self.tfidf_idf_model_path = self.model_dir / "tfidf_idf.model"

    # -------------------- 数据加载 --------------------
    # -------------------- 数据加载 --------------------
    def load_data(self):
        """
        加载训练集和测试集 CSV 文件，并开启多行文本容错机制防止错位
        :return: (train_df, test_df)
        """
        print("正在读取数据...")
        # 增加 multiLine, escape, quote 参数，完美解析带换行和双引号的评论文本！
        train_df = self.spark.read.csv(
            str(self.train_path), 
            header=True, 
            inferSchema=True,
            multiLine=True,    # 允许文本包含换行符
            escape='"',        # 转义文本内部的双引号
            quote='"'          # 将双引号包裹的内容视为整体
        )
        
        test_df = self.spark.read.csv(
            str(self.test_path), 
            header=True, 
            inferSchema=True,
            multiLine=True,
            escape='"',
            quote='"'
        )

        # 加载完毕后，立即进行数据质量校验
        self.check_data_quality(train_df, "Train Set")
        self.check_data_quality(test_df, "Test Set")

        return train_df, test_df
    # -------------------- 数据质量检验 --------------------
    def check_data_quality(self, df, dataset_name="Train"):
        """
        检验数据是否正常，排查列错位问题
        """
        print(f"\n================ 🔍 数据质量检验 ({dataset_name}) ================")
        
        # 1. 直观检查前几行数据（重点看 id, user_id, rating 有没有跑到别人的格子里）
        print(f"[{dataset_name}] 前 5 行数据预览:")
        df.show(5, truncate=True)
        
        # 2. 如果存在 rating 列，严格检查其分布
        if "rating" in df.columns:
            print(f"[{dataset_name}] rating 列的数值分布 (正常情况应该只有 1, 2, 3, 4, 5):")
            df.groupBy("rating").count().orderBy("rating").show(truncate=False)
            
            # 统计异常 rating 的数量
            # 将常见类型的 1-5 星级全部纳入白名单
            valid_ratings = ['1', '2', '3', '4', '5', '1.0', '2.0', '3.0', '4.0', '5.0', 1, 2, 3, 4, 5, 1.0, 2.0, 3.0, 4.0, 5.0]
            invalid_count = df.filter(~df["rating"].isin(valid_ratings)).count()
            
            if invalid_count > 0:
                print(f"🚨 警告: 发现 {invalid_count} 条数据的 rating 不是 1-5 分！数据可能依然存在错位。")
            else:
                print("✅ 完美！rating 列数据非常干净，未发现错位现象！")    
                print("==================================================================\n")
    def clean_text(self, df):
        """
        分开处理 title 和 comment，填充空值（空字符串），生成干净的 title 和 comment 列
        :param df: 包含 title, comment 的 DataFrame
        :return: 添加了 cleaned_title 和 cleaned_comment 列的 DataFrame（保留原列）
        """
        from pyspark.sql.functions import col, when, isnan, isnull, lit

        # 处理 title：空值或仅空格替换为空字符串
        df = df.withColumn(
            "cleaned_title",
            when(col("title").isNull(), lit(""))
            .otherwise(col("title"))
        )
        # 处理 comment：空值或仅空格替换为空字符串
        df = df.withColumn(
            "cleaned_comment",
            when(col("comment").isNull(), lit(""))
            .otherwise(col("comment"))
        )
        # 可选：移除首尾空格（但分词器会处理，不强制）
        return df

    def tokenize_and_remove_stopwords(self, df):
        """
        对 cleaned_title 和 cleaned_comment 列分别进行分词并移除停用词，
        生成 title_words_filtered 和 comment_words_filtered 两列（单词数组）
        :param df: 包含 cleaned_title 和 cleaned_comment 列的 DataFrame
        :return: 添加了 title_words_filtered 和 comment_words_filtered 列的 DataFrame
        """
        from pyspark.ml.feature import Tokenizer, StopWordsRemover

        # 1. 分词
        tokenizer = Tokenizer(inputCol="cleaned_title", outputCol="title_words")
        df = tokenizer.transform(df)
        tokenizer = Tokenizer(inputCol="cleaned_comment", outputCol="comment_words")
        df = tokenizer.transform(df)

        # 2. 移除停用词（使用默认英文停用词）
        remover = StopWordsRemover(inputCol="title_words", outputCol="title_words_filtered")
        df = remover.transform(df)
        remover = StopWordsRemover(inputCol="comment_words", outputCol="comment_words_filtered")
        df = remover.transform(df)

        # 可选：删除中间列以节省内存
        # df = df.drop("title_words", "comment_words")
        return df

    # -------------------- Word2Vec --------------------
    def _train_word2vec_model(self, train_words_df):
        """
        分别训练标题和评论的 Word2Vec 模型（参数固定：vectorSize=150, windowSize=7, minCount=5）
        优化：增加分区数、使用 MEMORY_AND_DISK 缓存、打印进度
        :param train_words_df: 训练集 DataFrame，包含 title_words_filtered 和 comment_words_filtered 列
        :return: (title_model, comment_model) 元组
        """
        from pyspark.ml.feature import Word2Vec
        from pyspark.storagelevel import StorageLevel

        print("准备训练 Word2Vec 模型...")
        # 只选择必要的单词列，并重新分区以平衡负载
        train_words_only = train_words_df.select("title_words_filtered", "comment_words_filtered")
        num_partitions = 200
        train_words_only = train_words_only.repartition(num_partitions)
        train_words_only.persist(StorageLevel.MEMORY_AND_DISK)
        train_words_only.count()  # 触发缓存并检查数据
        print(f"单词数据已缓存，分区数: {num_partitions}")

        # 训练标题的 Word2Vec 模型
        print("开始训练标题 Word2Vec 模型...")
        title_w2v = Word2Vec(
            vectorSize=150,
            windowSize=7,
            minCount=5,
            inputCol="title_words_filtered",
            outputCol="title_w2v_features",
            numPartitions=num_partitions,
            maxSentenceLength=1000
        )
        title_model = title_w2v.fit(train_words_only)
        print("标题 Word2Vec 模型训练完成")

        # 训练评论的 Word2Vec 模型
        print("开始训练评论 Word2Vec 模型...")
        comment_w2v = Word2Vec(
            vectorSize=150,
            windowSize=7,
            minCount=5,
            inputCol="comment_words_filtered",
            outputCol="comment_w2v_features",
            numPartitions=num_partitions,
            maxSentenceLength=1000
        )
        comment_model = comment_w2v.fit(train_words_only)
        print("评论 Word2Vec 模型训练完成")

        # 释放缓存
        train_words_only.unpersist()
        print("单词数据缓存已释放")

        return title_model, comment_model

    def _transform_word2vec(self, df, word2vec_models):
        """
        使用训练好的 Word2Vec 模型将标题和评论分别转换为向量列
        优化：增加进度打印
        :param df: 包含 title_words_filtered 和 comment_words_filtered 列的 DataFrame
        :param word2vec_models: (title_model, comment_model) 元组
        :return: 添加了 title_w2v_features 和 comment_w2v_features 列的 DataFrame
        """
        title_model, comment_model = word2vec_models

        print("正在应用标题 Word2Vec 模型...")
        df = title_model.transform(df)
        print("标题 Word2Vec 转换完成")

        print("正在应用评论 Word2Vec 模型...")
        df = comment_model.transform(df)
        print("评论 Word2Vec 转换完成")

        return df

    # -------------------- TF-IDF --------------------
    def _train_tfidf_model(self, train_words_df):
        """
        分别训练标题和评论的 TF-IDF 模型（CountVectorizer + IDF）
        :param train_words_df: 训练集 DataFrame，包含 title_words_filtered 和 comment_words_filtered 列
        :return: 元组 ((title_cv_model, title_idf_model), (comment_cv_model, comment_idf_model))
        """
        from pyspark.ml.feature import CountVectorizer, IDF

        print("开始训练标题 TF-IDF 模型...")
        title_cv = CountVectorizer(
            inputCol="title_words_filtered",
            outputCol="title_raw_features",
            vocabSize=22000,
            minDF=5
        )
        title_cv_model = title_cv.fit(train_words_df)
        title_raw_df = title_cv_model.transform(train_words_df)
        title_idf = IDF(inputCol="title_raw_features", outputCol="title_tfidf_features")
        title_idf_model = title_idf.fit(title_raw_df)

        print("开始训练评论 TF-IDF 模型...")
        comment_cv = CountVectorizer(
            inputCol="comment_words_filtered",
            outputCol="comment_raw_features",
            vocabSize=22000,
            minDF=5
        )
        comment_cv_model = comment_cv.fit(train_words_df)
        comment_raw_df = comment_cv_model.transform(train_words_df)
        comment_idf = IDF(inputCol="comment_raw_features", outputCol="comment_tfidf_features")
        comment_idf_model = comment_idf.fit(comment_raw_df)

        return (title_cv_model, title_idf_model), (comment_cv_model, comment_idf_model)

    def _transform_tfidf(self, df, tfidf_models):
        """
        使用训练好的 TF-IDF 模型将标题和评论分别转换为特征向量
        :param df: 包含 title_words_filtered 和 comment_words_filtered 列的 DataFrame
        :param tfidf_models: 元组 ((title_cv_model, title_idf_model), (comment_cv_model, comment_idf_model))
        :return: 添加了 title_tfidf_features 和 comment_tfidf_features 列的 DataFrame
        """
        (title_cv_model, title_idf_model), (comment_cv_model, comment_idf_model) = tfidf_models

        print("正在应用标题 TF-IDF 模型...")
        df = title_cv_model.transform(df)
        df = title_idf_model.transform(df)
        print("标题 TF-IDF 转换完成")

        print("正在应用评论 TF-IDF 模型...")
        df = comment_cv_model.transform(df)
        df = comment_idf_model.transform(df)
        print("评论 TF-IDF 转换完成")

        # 删除中间列以节省内存
        df = df.drop("title_raw_features", "comment_raw_features")
        return df

    # -------------------- 保存特征 --------------------
    def save_features(self, train_features_df, test_features_df):
        """
        将训练集和测试集的特征 DataFrame 保存为 Parquet 格式
        特征 DataFrame 应包含至少以下列：
            id, rating (仅训练集), title_w2v_features, comment_w2v_features,
            title_tfidf_features, comment_tfidf_features
        :param train_features_df: 训练集特征 DataFrame
        :param test_features_df: 测试集特征 DataFrame
        """
        # 保存训练集特征
        train_features_df.write.mode("overwrite").parquet(str(self.train_feature_path))
        # 保存测试集特征
        test_features_df.write.mode("overwrite").parquet(str(self.test_feature_path))
        print(f"训练集特征已保存至: {self.train_feature_path}")
        print(f"测试集特征已保存至: {self.test_feature_path}")

    def analyze_features(self, train_features_df):
        """
        分析 Word2Vec 和 TF-IDF 特征的统计分布（采样或全量）
        包括：
            - Word2Vec 各维度的均值、标准差、分位数
            - TF-IDF 稀疏向量的总体稀疏度、非零元素比例等
        统计结果保存为文本文件
        :param train_features_df: 训练集特征 DataFrame
        :return: 统计信息字典（可选）
        """
        import numpy as np
        from pyspark.sql.functions import col, udf, explode, lit, expr
        from pyspark.sql.types import ArrayType, DoubleType
        from pyspark.ml.linalg import DenseVector, SparseVector

        # 创建输出文件路径
        stats_file = self.feature_dir / "feature_stats.txt"

        with open(stats_file, "w") as f:
            f.write("========== 特征统计报告 ==========\n\n")

            # 1. 分析 Word2Vec 稠密向量（标题）
            f.write("【标题 Word2Vec 特征 (title_w2v_features)】\n")
            self._analyze_dense_vector(train_features_df, "title_w2v_features", f)

            f.write("\n【评论 Word2Vec 特征 (comment_w2v_features)】\n")
            self._analyze_dense_vector(train_features_df, "comment_w2v_features", f)

            f.write("\n【标题 TF-IDF 特征 (title_tfidf_features)】\n")
            self._analyze_sparse_vector(train_features_df, "title_tfidf_features", f)

            f.write("\n【评论 TF-IDF 特征 (comment_tfidf_features)】\n")
            self._analyze_sparse_vector(train_features_df, "comment_tfidf_features", f)

        print(f"特征统计报告已保存至: {stats_file}")

    def _analyze_dense_vector(self, df, col_name, file_handle):
        """
        辅助函数：分析稠密向量列（Word2Vec）的统计信息
        """
        from pyspark.sql.functions import col, explode, expr
        from pyspark.sql.types import ArrayType, DoubleType
        from pyspark.ml.linalg import DenseVector

        # 将 DenseVector 转换为数组
        to_array_udf = udf(lambda v: v.toArray().tolist() if v is not None else None, ArrayType(DoubleType()))
        df_arr = df.withColumn(f"{col_name}_arr", to_array_udf(col(col_name)))

        # 展开为每个维度一行
        df_exploded = df_arr.select(explode(f"{col_name}_arr").alias("value"))

        # 计算统计量
        stats = df_exploded.selectExpr(
            "mean(value) as mean",
            "stddev(value) as std",
            "min(value) as min",
            "max(value) as max",
            "percentile(value, 0.25) as q25",
            "percentile(value, 0.5) as median",
            "percentile(value, 0.75) as q75"
        ).collect()[0]

        file_handle.write(f"  均值: {stats.mean:.6f}\n")
        file_handle.write(f"  标准差: {stats.std:.6f}\n")
        file_handle.write(f"  最小值: {stats.min:.6f}\n")
        file_handle.write(f"  25%分位数: {stats.q25:.6f}\n")
        file_handle.write(f"  中位数: {stats.median:.6f}\n")
        file_handle.write(f"  75%分位数: {stats.q75:.6f}\n")
        file_handle.write(f"  最大值: {stats.max:.6f}\n")

    def _analyze_sparse_vector(self, df, col_name, file_handle):
        """
        辅助函数：分析稀疏向量列（TF-IDF）的统计信息
        """
        from pyspark.sql.functions import col, udf, size
        from pyspark.ml.linalg import SparseVector

        # 定义 UDF 获取非零元素数量
        get_nonzero_count = udf(lambda v: v.numNonzeros() if v is not None else 0)
        # 定义 UDF 获取向量维度
        get_dim = udf(lambda v: v.size if v is not None else 0)
        # 定义 UDF 获取非零比例
        get_sparsity = udf(lambda v: v.numNonzeros() / v.size if v is not None else 1.0)

        df_stats = df.select(
            get_nonzero_count(col(col_name)).alias("nonzero_count"),
            get_dim(col(col_name)).alias("dim"),
            get_sparsity(col(col_name)).alias("sparsity")
        )

        # 汇总统计
        stats = df_stats.selectExpr(
            "avg(nonzero_count) as avg_nonzero",
            "stddev(nonzero_count) as std_nonzero",
            "min(nonzero_count) as min_nonzero",
            "max(nonzero_count) as max_nonzero",
            "avg(dim) as avg_dim",
            "avg(sparsity) as avg_sparsity"
        ).collect()[0]

        file_handle.write(f"  向量维度: {stats.avg_dim:.0f}\n")
        file_handle.write(f"  平均非零元素数: {stats.avg_nonzero:.2f}\n")
        file_handle.write(f"  非零元素数标准差: {stats.std_nonzero:.2f}\n")
        file_handle.write(f"  最小非零元素数: {stats.min_nonzero:.0f}\n")
        file_handle.write(f"  最大非零元素数: {stats.max_nonzero:.0f}\n")
        file_handle.write(f"  平均稀疏度（非零比例）: {stats.avg_sparsity:.6f}\n")

    def compute_stats(self, df, numeric_cols, print_sample=True):
        """
        计算指定数值列的分位数、均值、标准差等统计量（安全版本）
        :param df: 训练集 DataFrame
        :param numeric_cols: 数值列名列表
        :param print_sample: 是否打印指定列的前10行（用于调试）
        :return: 字典，key 为列名，value 为统计量字典
        """
        from pyspark.sql.functions import mean, stddev, min, max, count, col
        from pyspark.sql.types import DoubleType
        import json

        # 调试：打印指定列的前10行
        if print_sample and numeric_cols:
            print("=== 调试信息：前10行 ===")
            df.show(10, truncate=False)
        # 定义安全转换 UDF
        from pyspark.sql.functions import udf
        def safe_to_double(v):
            try:
                return float(v) if v is not None else None
            except (ValueError, TypeError):
                return None
        safe_to_double_udf = udf(safe_to_double, DoubleType())

        stats_dict = {}
        for col_name in numeric_cols:
            if col_name not in df.columns:
                print(f"警告: {col_name} 列不存在，跳过")
                continue

            # 安全转换
            safe_col = safe_to_double_udf(col(col_name)).alias("tmp")
            temp_df = df.select(safe_col)

            # 统计基本统计量
            row = temp_df.select(
                mean("tmp").alias("mean"),
                stddev("tmp").alias("std"),
                min("tmp").alias("min"),
                max("tmp").alias("max"),
                count("tmp").alias("count")
            ).collect()[0]

            # 如果有效数据太少，跳过分位数计算
            if row["count"] == 0:
                print(f"警告: {col_name} 列转换后无有效数值，跳过")
                continue

            # 计算分位数（使用安全转换后的列）
            df_with_safe = df.withColumn("tmp_safe", safe_to_double_udf(col(col_name)))
            try:
                quantiles = df_with_safe.approxQuantile("tmp_safe", [0.25, 0.5, 0.75], 0.01)
                q25, median, q75 = quantiles if quantiles else (None, None, None)
            except Exception as e:
                print(f"计算 {col_name} 分位数时出错: {e}")
                q25 = median = q75 = None

            stats_dict[col_name] = {
                "mean": row["mean"],
                "std": row["std"],
                "min": row["min"],
                "max": row["max"],
                "count": row["count"],
                "q25": q25,
                "median": median,
                "q75": q75
            }

        # 保存统计结果
        stats_file = self.feature_dir / "numeric_stats.json"
        with open(stats_file, "w") as f:
            json.dump(stats_dict, f, indent=2, default=str)
        print(f"数值特征统计结果已保存至: {stats_file}")

        return stats_dict

    def plot_distributions(self, train_features_df, sample_frac=0.01):
        """
        绘制特征分布图：
            - 数值特征（votes, time）的直方图与箱线图
            - Word2Vec 特征降维后的散点图（PCA）
            - TF-IDF 非零元素数量分布图
        """
        import matplotlib.pyplot as plt
        import numpy as np
        import pandas as pd
        from sklearn.decomposition import PCA as SklearnPCA
        from pyspark.ml.linalg import DenseVector, SparseVector

        # 采样
        sample_count = train_features_df.count()
        if sample_count > 10000:
            sample_df = train_features_df.sample(fraction=sample_frac, seed=42).limit(10000)
        else:
            sample_df = train_features_df
        sample_pd = sample_df.toPandas()
        print(f"采样数据量: {len(sample_pd)} 行")
        print("DataFrame 列名:", sample_pd.columns.tolist())

        # 1. 数值特征
        numeric_cols = ["votes", "time"]
        available_numeric_cols = []
        for col_name in numeric_cols:
            if col_name in sample_pd.columns:
                # 检查并转换类型
                if not pd.api.types.is_numeric_dtype(sample_pd[col_name]):
                    print(f"警告: {col_name} 列不是数值类型，尝试转换")
                    sample_pd[col_name] = pd.to_numeric(sample_pd[col_name], errors='coerce')
                # 检查转换后是否有数值
                if sample_pd[col_name].notna().any():
                    available_numeric_cols.append(col_name)
                else:
                    print(f"警告: {col_name} 列转换后全为 NaN，跳过")
            else:
                print(f"警告: {col_name} 列不存在，跳过")

        if not available_numeric_cols:
            print("警告: 未找到可用的数值列，跳过数值特征绘图")
        else:
            for col_name in available_numeric_cols:
                try:
                    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
                    # 去除 NaN 值
                    data = sample_pd[col_name].dropna()
                    if len(data) == 0:
                        print(f"警告: {col_name} 列无有效数据，跳过")
                        plt.close(fig)
                        continue
                    data.hist(bins=50, ax=axes[0])
                    axes[0].set_title(f"{col_name} Histogram")
                    data.plot.box(ax=axes[1])
                    axes[1].set_title(f"{col_name} Boxplot")
                    plt.tight_layout()
                    plt.savefig(self.fig_dir / f"{col_name}_dist.png", dpi=100)
                    plt.close()
                    print(f"已保存 {col_name} 分布图")
                except Exception as e:
                    print(f"绘制 {col_name} 分布图时出错: {e}")
                    plt.close('all')

        # 2. Word2Vec PCA
        w2v_cols = ["title_w2v_features", "comment_w2v_features"]
        for w2v_col in w2v_cols:
            if w2v_col not in sample_pd.columns:
                print(f"警告: {w2v_col} 列不存在，跳过")
                continue
            # 提取向量并转换为数组
            vec_series = sample_pd[w2v_col].dropna()
            if len(vec_series) == 0:
                print(f"警告: {w2v_col} 列无有效向量，跳过")
                continue
            # 转换为列表的列表
            vec_list = []
            valid_idx = []
            for idx, v in vec_series.items():
                if isinstance(v, DenseVector):
                    vec_list.append(v.toArray().tolist())
                    valid_idx.append(idx)
                else:
                    print(f"警告: {w2v_col} 第 {idx} 行不是 DenseVector，跳过")
            if len(vec_list) < 2:
                print(f"警告: {w2v_col} 有效样本不足 2，跳过")
                continue
            X = np.array(vec_list)
            try:
                pca = SklearnPCA(n_components=2)
                X_pca = pca.fit_transform(X)
            except Exception as e:
                print(f"PCA 降维失败: {e}")
                continue

            # 获取对应的 rating，并转换为数值
            if "rating" in sample_pd.columns:
                ratings_series = sample_pd.loc[valid_idx, "rating"]
                ratings_series = pd.to_numeric(ratings_series, errors='coerce')
                ratings = ratings_series.values
                # 如果全为 NaN，则不显示颜色条
                if np.isnan(ratings).all():
                    ratings = None
            else:
                ratings = None

            plt.figure(figsize=(8, 6))
            if ratings is not None:
                scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=ratings, cmap='viridis', alpha=0.5)
                plt.colorbar(scatter, label='Rating')
            else:
                plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5)
            plt.title(f"{w2v_col} PCA Projection")
            plt.xlabel("PC1")
            plt.ylabel("PC2")
            plt.savefig(self.fig_dir / f"{w2v_col}_pca.png", dpi=100)
            plt.close()
            print(f"已保存 {w2v_col} PCA 图")

        # 3. TF-IDF 非零元素分布
        tfidf_cols = ["title_tfidf_features", "comment_tfidf_features"]
        for tfidf_col in tfidf_cols:
            if tfidf_col not in sample_pd.columns:
                print(f"警告: {tfidf_col} 列不存在，跳过")
                continue
            # 计算非零个数
            nonzero_counts = sample_pd[tfidf_col].apply(
                lambda v: v.numNonzeros() if isinstance(v, SparseVector) else 0
            )
            # 过滤掉 0 的（可选）
            nonzero_counts = nonzero_counts[nonzero_counts > 0]
            if len(nonzero_counts) == 0:
                print(f"警告: {tfidf_col} 列无非零元素，跳过")
                continue
            plt.figure(figsize=(8, 4))
            plt.hist(nonzero_counts, bins=50, alpha=0.7, edgecolor='black')
            plt.title(f"{tfidf_col} Nonzero Count Distribution")
            plt.xlabel("Number of non-zero entries")
            plt.ylabel("Frequency")
            plt.savefig(self.fig_dir / f"{tfidf_col}_nonzero.png", dpi=100)
            plt.close()
            print(f"已保存 {tfidf_col} 非零元素分布图")

    # -------------------- 主流程 --------------------
    def train_w2v(self):
        """训练 Word2Vec 模型并保存特征"""
        w2v_train_path = self.feature_dir / "w2v_train.parquet"
        w2v_test_path = self.feature_dir / "w2v_test.parquet"

        if not self.force_recompute and w2v_train_path.exists() and w2v_test_path.exists():
            print("Word2Vec 特征已存在，跳过训练。")
            return

        # 1. 加载数据
        train_df, test_df = self.load_data()

        # 2. 清洗与预处理
        train_clean = self.clean_text(train_df)
        test_clean = self.clean_text(test_df)
        train_clean = self.tokenize_and_remove_stopwords(train_clean)
        test_clean = self.tokenize_and_remove_stopwords(test_clean)
        train_clean.cache()
        test_clean.cache()

        # 3. 训练模型
        print("开始训练 Word2Vec 模型...")
        title_model, comment_model = self._train_word2vec_model(train_clean)

        # 4. 转换特征
        print("转换训练集 Word2Vec 特征...")
        train_features = self._transform_word2vec(train_clean, (title_model, comment_model))
        train_features = train_features.select("id", "title_w2v_features", "comment_w2v_features")

        print("转换测试集 Word2Vec 特征...")
        test_features = self._transform_word2vec(test_clean, (title_model, comment_model))
        test_features = test_features.select("id", "title_w2v_features", "comment_w2v_features")

        # 5. 保存特征
        train_features.write.mode("overwrite").parquet(str(w2v_train_path))
        test_features.write.mode("overwrite").parquet(str(w2v_test_path))
        print("Word2Vec 特征保存完成。")

        # 6. 释放缓存
        train_clean.unpersist()
        test_clean.unpersist()

        # 7. 保存模型（可选）
        title_model.write().overwrite().save(str(self.model_dir / "title_w2v.model"))
        comment_model.write().overwrite().save(str(self.model_dir / "comment_w2v.model"))

    def train_tfidf(self):
        """训练 TF-IDF 模型并保存特征"""
        tfidf_train_path = self.feature_dir / "tfidf_train.parquet"
        tfidf_test_path = self.feature_dir / "tfidf_test.parquet"

        if not self.force_recompute and tfidf_train_path.exists() and tfidf_test_path.exists():
            print("TF-IDF 特征已存在，跳过训练。")
            return

        # 1. 加载数据
        train_df, test_df = self.load_data()

        # 2. 清洗与预处理（与 Word2Vec 独立，可复用缓存但为简单起见重新处理）
        train_clean = self.clean_text(train_df)
        test_clean = self.clean_text(test_df)
        train_clean = self.tokenize_and_remove_stopwords(train_clean)
        test_clean = self.tokenize_and_remove_stopwords(test_clean)
        train_clean.cache()
        test_clean.cache()

        # 3. 训练模型
        print("开始训练 TF-IDF 模型...")
        (title_cv, title_idf), (comment_cv, comment_idf) = self._train_tfidf_model(train_clean)

        # 4. 转换特征
        print("转换训练集 TF-IDF 特征...")
        train_features = self._transform_tfidf(train_clean, ((title_cv, title_idf), (comment_cv, comment_idf)))
        train_features = train_features.select("id", "title_tfidf_features", "comment_tfidf_features")

        print("转换测试集 TF-IDF 特征...")
        test_features = self._transform_tfidf(test_clean, ((title_cv, title_idf), (comment_cv, comment_idf)))
        test_features = test_features.select("id", "title_tfidf_features", "comment_tfidf_features")

        # 5. 保存特征
        train_features.write.mode("overwrite").parquet(str(tfidf_train_path))
        test_features.write.mode("overwrite").parquet(str(tfidf_test_path))
        print("TF-IDF 特征保存完成。")

        # 6. 释放缓存
        train_clean.unpersist()
        test_clean.unpersist()

        # 7. 保存模型
        title_cv.write().overwrite().save(str(self.model_dir / "title_cv.model"))
        title_idf.write().overwrite().save(str(self.model_dir / "title_idf.model"))
        comment_cv.write().overwrite().save(str(self.model_dir / "comment_cv.model"))
        comment_idf.write().overwrite().save(str(self.model_dir / "comment_idf.model"))
    def merge_features(self):
        """合并原始其他列与 Word2Vec、TF-IDF 特征，生成最终大表"""
        final_train_path = self.feature_dir / "final_train.parquet"
        final_test_path = self.feature_dir / "final_test.parquet"

        if not self.force_recompute and final_train_path.exists() and final_test_path.exists():
            print("最终特征表已存在，跳过合并。")
            return

        # 1. 加载原始数据，去除 title 和 comment
        train_raw, test_raw = self.load_data()
        train_other = train_raw.drop("title", "comment")
        test_other = test_raw.drop("title", "comment")
        print("train_raw.columns:")
        print(train_raw.columns)


        # 2. 加载 Word2Vec 特征
        w2v_train = self.spark.read.parquet(str(self.feature_dir / "w2v_train.parquet"))
        w2v_test = self.spark.read.parquet(str(self.feature_dir / "w2v_test.parquet"))

        # 3. 加载 TF-IDF 特征
        tfidf_train = self.spark.read.parquet(str(self.feature_dir / "tfidf_train.parquet"))
        tfidf_test = self.spark.read.parquet(str(self.feature_dir / "tfidf_test.parquet"))

        # 4. 通过 id 连接
        train_merged = train_other.join(w2v_train, on="id").join(tfidf_train, on="id")
        test_merged = test_other.join(w2v_test, on="id").join(tfidf_test, on="id")
        print("train_merged.columns")
        print(train_merged.columns)

        # 5. 保存
        train_merged.write.mode("overwrite").parquet(str(final_train_path))
        test_merged.write.mode("overwrite").parquet(str(final_test_path))
        print("最终特征表保存完成。")
    def generate_figures(self):
        """对最终大表进行分析与可视化"""
        if not self.force_recreate_figs:
            print("图表生成被跳过。")
            return

        final_train_path = self.feature_dir / "final_train.parquet"
        if not final_train_path.exists():
            print("最终训练特征表不存在，请先运行 merge_features。")
            return

        final_train = self.spark.read.parquet(str(final_train_path))
        self.plot_distributions(final_train, sample_frac=0.01)
        self.compute_stats(final_train, numeric_cols=["votes"])


if __name__ == "__main__":
    # 假设第一个格子已运行，存在 spark, TRAIN_PATH, TEST_PATH, FEATURE_DIR, MODEL_DIR, FIG_DIR,
    # FORCE_RECOMPUTE_FEATURES, FORCE_RETRAIN, FORCE_RECREATE_FIGS

    preparer = ReviewPrepare(
        spark=spark,
        train_path=TRAIN_PATH,
        test_path=TEST_PATH,
        feature_dir=FEATURE_DIR,
        model_dir=MODEL_DIR,
        fig_dir=FIG_DIR,
        force_recompute=FORCE_RECOMPUTE_FEATURES,
        force_retrain=FORCE_RETRAIN,
        force_recreate_figs=FORCE_RECREATE_FIGS
    )

    preparer.train_w2v()
    preparer.train_tfidf()
    preparer.merge_features()
    preparer.generate_figures()

In [ ]:
# 诊断代码：分析 final_train.parquet 中 Word2Vec 向量列的类型
from pyspark.sql import SparkSession
from pyspark.ml.linalg import DenseVector, SparseVector
import pandas as pd
import numpy as np

# 加载最终特征表
final_train_path = "./featured_data/final_train.parquet"
df = spark.read.parquet(final_train_path)

# 采样
sample_df = df.sample(fraction=0.01, seed=42).limit(10000)
sample_pd = sample_df.toPandas()

print("采样数据量:", len(sample_pd))

# 检查 title_w2v_features 列的类型分布
vec_series = sample_pd['title_w2v_features']
type_counts = vec_series.apply(lambda x: type(x).__name__).value_counts()
print("title_w2v_features 类型分布:\n", type_counts)

# 检查 comment_w2v_features 列
vec_series2 = sample_pd['comment_w2v_features']
type_counts2 = vec_series2.apply(lambda x: type(x).__name__).value_counts()
print("\ncomment_w2v_features 类型分布:\n", type_counts2)

# 如果不是 DenseVector，显示一些示例
non_dense = vec_series[vec_series.apply(lambda x: not isinstance(x, DenseVector))]
if len(non_dense) > 0:
    print(f"\n非 DenseVector 示例（前5个）:")
    for idx, val in non_dense.head(5).items():
        print(f"  行 {idx}: 类型={type(val)}")
        if isinstance(val, SparseVector):
            print(f"    稀疏向量大小: {val.size}, 非零元素数: {val.numNonzeros()}")
        elif isinstance(val, (list, np.ndarray)):
            print(f"    列表/数组长度: {len(val)}")
        else:
            print(f"    值: {val}")

In [ ]:
import os
import zipfile
from pathlib import Path

# ========== 配置 ==========
# 项目根目录（当前工作目录）
project_root = Path.cwd()  # 如果不准，可以直接写死绝对路径，如 Path(".")

# 要打包的文件夹名称（相对于 project_root）
# 如果希望打包整个项目，设置为 ""，表示 project_root 本身
folder_to_zip = ""  

# 输出 zip 文件名
zip_name = "Comp5434_project.zip"
zip_path = project_root / zip_name

# 排除的文件夹/文件（名称，不包含路径）
# 本地运行建议额外排除 .git、虚拟环境（venv/env/anaconda3等）
exclude_dirs = {"data", "__pycache__", ".ipynb_checkpoints", ".git", "venv", "env"}
# 必须把生成的 zip 文件名也排除掉，否则 os.walk 扫到一半会把它自己也打包进去！
exclude_files = {".gitignore", ".DS_Store", zip_name} 


# ========== 打包逻辑 ==========
def zipdir(path, ziph, exclude_dirs, exclude_files):
    for root, dirs, files in os.walk(path):
        # 排除文件夹（就地修改 dirs 列表，这样 os.walk 就不会进入这些目录了）
        dirs[:] = [d for d in dirs if d not in exclude_dirs]
        for file in files:
            if file in exclude_files:
                continue
            file_path = os.path.join(root, file)
            # 计算相对路径，避免在 zip 压缩包中出现类似于 ./home 这样的绝对路径层级
            arcname = os.path.relpath(file_path, path)
            ziph.write(file_path, arcname)


if __name__ == "__main__":
    # 确定要打包的路径
    target_path = project_root / folder_to_zip if folder_to_zip else project_root

    print(f"🚀 开始打包: {target_path} ...")
    
    # 切换到目标路径打包更稳妥
    original_dir = os.getcwd()
    try:
        os.chdir(project_root)
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipdir(".", zipf, exclude_dirs, exclude_files)
    finally:
        # 无论发生什么报错，保证切回原目录
        os.chdir(original_dir)

    print("✅ 打包完成！")
    print(f"📦 文件位置: {zip_path}")
    print(f"⚖️ 文件大小: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

In [ ]:
# 对文本特征进行数值分析
from pyspark.sql import SparkSession
from pyspark.ml.linalg import DenseVector, SparseVector
import pandas as pd
import numpy as np
import json

# 路径（根据实际情况修改）
final_train_path = "./featured_data/final_train.parquet"
output_stats_path = "./featured_data/text_feature_stats.json"

# 采样比例（可调整）
sample_frac = 0.01
max_rows = 10000

# 读取数据
df = spark.read.parquet(final_train_path)
total_rows = df.count()
print(f"总行数: {total_rows}")

# 采样
if total_rows > max_rows:
    sample_df = df.sample(fraction=sample_frac, seed=42).limit(max_rows)
else:
    sample_df = df
sample_pd = sample_df.toPandas()
print(f"采样行数: {len(sample_pd)}")

# 定义分析函数
def analyze_w2v_vectors(vectors, name):
    """分析稠密向量列"""
    # 提取所有向量为数组
    valid = vectors.dropna()
    if len(valid) == 0:
        print(f"{name}: 无有效数据")
        return None
    arr = np.array([v.toArray() if isinstance(v, DenseVector) else v.toArray() for v in valid])
    # 按行统计（每个向量）
    row_means = arr.mean(axis=1)
    row_stds = arr.std(axis=1)
    row_mins = arr.min(axis=1)
    row_maxs = arr.max(axis=1)
    # 按列统计（每个维度）
    col_means = arr.mean(axis=0)
    col_stds = arr.std(axis=0)
    col_mins = arr.min(axis=0)
    col_maxs = arr.max(axis=0)
    # 汇总
    stats = {
        "总向量数": len(valid),
        "向量维度": arr.shape[1],
        "向量均值（整体）": float(arr.mean()),
        "向量标准差（整体）": float(arr.std()),
        "向量最大值（整体）": float(arr.max()),
        "向量最小值（整体）": float(arr.min()),
        "各向量均值分布": {
            "mean": float(row_means.mean()),
            "std": float(row_means.std()),
            "min": float(row_means.min()),
            "25%": float(np.percentile(row_means, 25)),
            "50%": float(np.percentile(row_means, 50)),
            "75%": float(np.percentile(row_means, 75)),
            "max": float(row_means.max())
        },
        "各向量标准差分布": {
            "mean": float(row_stds.mean()),
            "std": float(row_stds.std()),
            "min": float(row_stds.min()),
            "25%": float(np.percentile(row_stds, 25)),
            "50%": float(np.percentile(row_stds, 50)),
            "75%": float(np.percentile(row_stds, 75)),
            "max": float(row_stds.max())
        },
        "各维度均值分布": {
            "mean": float(col_means.mean()),
            "std": float(col_means.std()),
            "min": float(col_means.min()),
            "25%": float(np.percentile(col_means, 25)),
            "50%": float(np.percentile(col_means, 50)),
            "75%": float(np.percentile(col_means, 75)),
            "max": float(col_means.max())
        },
        "各维度标准差分布": {
            "mean": float(col_stds.mean()),
            "std": float(col_stds.std()),
            "min": float(col_stds.min()),
            "25%": float(np.percentile(col_stds, 25)),
            "50%": float(np.percentile(col_stds, 50)),
            "75%": float(np.percentile(col_stds, 75)),
            "max": float(col_stds.max())
        }
    }
    return stats

def analyze_tfidf_vectors(vectors, name):
    """分析稀疏向量列"""
    valid = vectors.dropna()
    if len(valid) == 0:
        print(f"{name}: 无有效数据")
        return None
    # 提取非零元素数量
    nnz = np.array([v.numNonzeros() for v in valid])
    # 提取每个向量的非零元素值（用于整体值分布）
    all_vals = []
    for v in valid:
        if v.numNonzeros() > 0:
            all_vals.extend(v.values)
    all_vals = np.array(all_vals)
    stats = {
        "总向量数": len(valid),
        "向量维度": valid.iloc[0].size if len(valid) > 0 else 0,
        "非零元素统计": {
            "总非零元素数": int(nnz.sum()),
            "平均每向量非零数": float(nnz.mean()),
            "标准差": float(nnz.std()),
            "最小值": int(nnz.min()),
            "25%分位数": float(np.percentile(nnz, 25)),
            "中位数": float(np.percentile(nnz, 50)),
            "75%分位数": float(np.percentile(nnz, 75)),
            "最大值": int(nnz.max())
        },
        "非零值分布": {
            "均值": float(all_vals.mean()),
            "标准差": float(all_vals.std()),
            "最小值": float(all_vals.min()),
            "25%分位数": float(np.percentile(all_vals, 25)),
            "中位数": float(np.percentile(all_vals, 50)),
            "75%分位数": float(np.percentile(all_vals, 75)),
            "最大值": float(all_vals.max())
        } if len(all_vals) > 0 else {},
        "稀疏度": {
            "平均稀疏度": float(nnz.mean() / (valid.iloc[0].size if len(valid) > 0 else 1)),
            "非零向量比例": float((nnz > 0).sum() / len(valid))
        }
    }
    return stats

# 分析各列
results = {}
for col_name in ["title_w2v_features", "comment_w2v_features"]:
    if col_name in sample_pd.columns:
        print(f"\n分析 {col_name} ...")
        results[col_name] = analyze_w2v_vectors(sample_pd[col_name], col_name)
    else:
        print(f"警告: {col_name} 列不存在")

for col_name in ["title_tfidf_features", "comment_tfidf_features"]:
    if col_name in sample_pd.columns:
        print(f"\n分析 {col_name} ...")
        results[col_name] = analyze_tfidf_vectors(sample_pd[col_name], col_name)
    else:
        print(f"警告: {col_name} 列不存在")

# 打印结果
print("\n========== 数值分析结果 ==========")
for name, stats in results.items():
    if stats is None:
        continue
    print(f"\n--- {name} ---")
    if "向量维度" in stats:
        print(f"向量维度: {stats['向量维度']}")
    if "总向量数" in stats:
        print(f"总向量数: {stats['总向量数']}")
    if "向量均值（整体）" in stats:
        print(f"整体均值: {stats['向量均值（整体）']:.6f}")
        print(f"整体标准差: {stats['向量标准差（整体）']:.6f}")
        print(f"整体最小值: {stats['向量最小值（整体）']:.6f}")
        print(f"整体最大值: {stats['向量最大值（整体）']:.6f}")
    if "各向量均值分布" in stats:
        print(f"各向量均值: 均值={stats['各向量均值分布']['mean']:.6f}, 标准差={stats['各向量均值分布']['std']:.6f}")
    if "非零元素统计" in stats:
        print(f"非零元素统计: 平均每向量 {stats['非零元素统计']['平均每向量非零数']:.2f}, 中位数 {stats['非零元素统计']['中位数']:.2f}")
    if "稀疏度" in stats:
        print(f"稀疏度: {stats['稀疏度']['平均稀疏度']:.6f}, 非零向量比例 {stats['稀疏度']['非零向量比例']:.4f}")

# 保存到文件
with open(output_stats_path, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n详细统计已保存至: {output_stats_path}")